# UppASD Python interface

This notebook explains the Python interface to UppASD provided in `mammos_spindynamics.uppasd` in detail.

In [3]:
import mammos_spindynamics
from mammos_spindynamics import uppasd

## Running simulations

:::{warning}
In order to keep the runtime of this notebook short we use unreasonably small values for `ncell`, `ip_mcnstep` and `mcnstep`.

For meaningfull simulations you will have to use higher values, suitable starting points could be `ncell 32 32 32`, `ip_mcnstep 25000`, `mcnstep 50000`
:::

All simulations in this notebook will be performed for Co2Fe2H4 for which the required static inputs are available in the database of `mammos_spindynamics`, as shown in the [quickstart notebook](quickstart.ipynb#Running-spindynamics-simulations-–-mammos_spindynamics.uppasd):

In [4]:
Co2Fe2H4_data = mammos_spindynamics.db.get_material(chemical_formula="Co2Fe2H4")
Co2Fe2H4_data.available_files

['jfile', 'momfile', 'posfile']

### Running simulations from fresh

The main input file for UppASD is called `inpsd.dat`, details about this file can be found in the [UppASD documentation](https://uppasd.github.io/UppASD-manual).

The Python interface in this package can create these input files. Currently, it has a number of assumptions hard-coded:
- we always run Monte Carlo simulations for the initial and the measurement phase (`ip_mode M` and `mode M`)
- a few other parameters are hard-coded (details below)
- anealing in the initial phase is not supported
- only `exchange`, `momfile` and `posfile` are supported as external files

The main object to control simulations is `uppasd.Simulation`. We can first get a list of required parameters for the input file:

In [6]:
uppasd.Simulation().required_parameters

{'alat', 'cell', 'initmag', 'ip_temp', 'maptype', 'posfiletype', 'temp'}

These parameters depend heavily on the simulation requirements and the the format of the additional input files and have to be provided by the user.

A few more parameters can be controlled by the user but have sensible defaults. To get a list of all available parameters run:

In [7]:
uppasd.Simulation().allowed_parameters

{'alat',
 'cell',
 'initmag',
 'ip_mcnstep',
 'ip_temp',
 'maptype',
 'mcnstep',
 'ncell',
 'posfiletype',
 'temp'}

We run simulations in two steps:
1. we create a `Simulation` object
2. we call a method of that object to run a simulation, currently `run` is the only method

We have to provude all required arguments by the time we call `run`. We can either pass them when creating the object or when calling the `run` method (we can freely choose for each parameter). Parameters passed to the `run` method are treated special and can simplify accessing the outputs later.

Therefor as a rule of thumb:
- pass all parameters that do not change when creating the object (e.g. cell, alat, ...)
- pass all parameters that you vary when calling `run` (e.g. ip_temp/temp, ncell, ...)

In addition to these parameters we also need to pass paths to three files:
- `exchanges` containing the exchange coupling constants
- `posfile` containing the position of the atoms in the unit cell
- `momfile` containing the magnetic moments of the atoms in the unit cell

These files will be copied into the run directory (more on that directory below) and in the resulting input file relative paths will be used. Note: the files will be renamed to `exchange`, `posfile`, and `momfile`, the original names are lost.

We now create a `Simulation` object. In this notebook we are only interested in varying temperature, so we pass all other parameters now:

In [16]:
simulation = uppasd.Simulation(
    cell=[[1.0, -0.5, 0], [0, 0.866, 0], [0, 0, 3.228]],
    alat=2.65e-10,
    ncell=(25, 25, 25),  # to keep the simulations short for this notebook
    posfile=Co2Fe2H4_data.posfile,
    posfiletype="D",  # we have to know this, the database currently does not provide it
    momfile=Co2Fe2H4_data.momfile,
    maptype=2,  # we have to know this, the database currently does not provide it
    exchange=Co2Fe2H4_data.jfile,  # the exchange file is always called 'jfile' in the database
    initmag=3,
    ip_mcnstep=250,  # to keep the simulations short in this notebook
    mcnstep=500,  # to keep the simulations short in this notebook
)
simulation  # TODO proper repr

We can now call the `run` method to actually perform a simulation. We need to pass a directory name where the output will be written to and the missing parameters. For temperature we can use a convenience: we can pass `T` and the same value will be used for `ip_temp` and `temp`.

In [17]:
sim_result = simulation.run("Co2Fe2H4", T=20)

Running UppASD in Co2Fe2H4/run-1 ... simulation finished, took 0:00:05


The `run` method displays some status information, telling us where the output has been written to and how long the simulation took. We will discuss analysing data in more detail in the next section.

We can also get the input file that would be run by calling the method `create_input_files`. This method returns two objects:
1. a string containing the content for `inpsd.dat`
2. a dictionary of auxilary files that will be copied to the run directory; keys will be the file names in the run directory, values are the original paths

We need to pass parameters similar to the run method.

In [21]:
inp_content, aux_files = simulation.create_input_files("Co2Fe2H4", T=20)

In [22]:
print(inp_content)

cell  1.0 -0.5 0.0
      0.0 0.866 0.0
      0.0 0.0 3.228
alat  2.65e-10
ncell 25 25 25
bc    P P P
sym   0

posfile     ./posfile
posfiletype D
momfile     ./momfile
maptype     2
exchange    ./jfile

initmag 3


ip_mode    M
ip_temp    20
ip_mcnstep 250

mode    M
temp    20
mcnstep 500

plotenergy   1
do_proj_avrg Y
do_cumu      Y



In [23]:
aux_files

{'exchange': PosixPath('/home/mlang/repos/mammos/mammos-devtools/packages/mammos-spindynamics/src/mammos_spindynamics/data/0001/jfile'),
 'posfile': PosixPath('/home/mlang/repos/mammos/mammos-devtools/packages/mammos-spindynamics/src/mammos_spindynamics/data/0001/posfile'),
 'momfile': PosixPath('/home/mlang/repos/mammos/mammos-devtools/packages/mammos-spindynamics/src/mammos_spindynamics/data/0001/momfile')}

We can overwrite parameters defined when creating the object at the time where we call the `run` method (or equally `create_input_files`). We demonstrate this here for `ncell` and `mcnstep`. We also pass two different temperatures:

In [26]:
inp_content, _ = simulation.create_input_files(
    out="Co2Fe2H4",
    ncell=(36, 36, 36),
    mcnstep=50_000,
    ip_temp=50,
    temp=55,
)
print(inp_content)

cell  1.0 -0.5 0.0
      0.0 0.866 0.0
      0.0 0.0 3.228
alat  2.65e-10
ncell 36 36 36
bc    P P P
sym   0

posfile     ./posfile
posfiletype D
momfile     ./momfile
maptype     2
exchange    ./jfile

initmag 3


ip_mode    M
ip_temp    50
ip_mcnstep 250

mode    M
temp    55
mcnstep 50000

plotenergy   1
do_proj_avrg Y
do_cumu      Y



UppASD can use a restart file from a previous simulation as starting point for the new simulation. To do that, we need to change `initmag 4` and we need to pass an additional `restartfile`. We can use the object returned from the `run` above to restart from that run. Again we show just the modified input file:

In [26]:
inp_content, _ = simulation.create_input_files(
    out="Co2Fe2H4",
    T=20,
    initmag=4,
    restartfile=sim_result.restart_file,
)
print(inp_content)

cell  1.0 -0.5 0.0
      0.0 0.866 0.0
      0.0 0.0 3.228
alat  2.65e-10
ncell 36 36 36
bc    P P P
sym   0

posfile     ./posfile
posfiletype D
momfile     ./momfile
maptype     2
exchange    ./jfile

initmag 3


ip_mode    M
ip_temp    50
ip_mcnstep 250

mode    M
temp    55
mcnstep 50000

plotenergy   1
do_proj_avrg Y
do_cumu      Y



### Starting from an existing `inpsd.dat` file

## Analysing simulation output

In [ ]:
uppasd.read(...)